We first begin by joining together the weather and flight datasets.

In [2]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
con = duckdb.connect('/content/drive/My Drive/CIS2450 Final Project/Models/flight_data.duckdb')
con.execute("ATTACH '/content/drive/My Drive/CIS2450 Final Project/Models/weather_data.duckdb' AS weather_db")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
query = """
SELECT
    f.MONTH,
    f.DAY_OF_MONTH,
    f.DISTANCE,
    CAST(f.CRS_DEP_TIME / 100 AS BIGINT) AS DEP_HOUR,

    -- All Weather Features included with logical imputations
    COALESCE(w.WIND_SPEED, 0) AS WIND_SPEED,
    COALESCE(w.WIND_GUST, 0) AS WIND_GUST,
    COALESCE(w.WIND_ANGLE, 0) AS WIND_ANGLE,
    COALESCE(w.VISIBILITY, 10) AS VISIBILITY,
    COALESCE(w.AIR_TEMP, 20) AS AIR_TEMP,
    COALESCE(w.DEW_POINT_TEMP, w.AIR_TEMP) AS DEW_POINT_TEMP, -- Default to air temp if missing
    COALESCE(w.PRESSURE, 1013.25) AS PRESSURE, -- Standard atmospheric pressure
    COALESCE(w.PRECIPITATION, 0) AS PRECIPITATION,
    COALESCE(w.CEILING_HEIGHT, 99999) AS CEILING_HEIGHT, -- 99999 implies infinite/clear ceiling

    -- Boolean flags
    COALESCE(w.HAS_THUNDERSTORM, 0) AS HAS_THUNDERSTORM,
    COALESCE(w.HAS_GUST, 0) AS HAS_GUST,
    COALESCE(w.HAS_CLOUDS, 0) AS HAS_CLOUDS,

    f.ARRIVAL_DELAYED AS target
FROM flights_cleaned f
LEFT JOIN weather_db.weather_cleaned w
    ON f.ORIGIN = w.STATION
    AND f.YEAR = w.YEAR
    AND f.MONTH = w.MONTH
    AND f.DAY_OF_MONTH = w.DAY
    AND CAST(f.CRS_DEP_TIME / 100 AS BIGINT) = w.HOUR
WHERE f.ARRIVAL_DELAYED IS NOT NULL
  AND f.CANCELLED = 0
"""
# Fetch the data
df = con.execute(query).fetchdf()

# Drop rows with any NaNs (including our problematic dew points)
df = df.dropna()

print(f"Cleaned dataset shape: {df.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cleaned dataset shape: (11565665, 17)


In [5]:
if torch.cuda.is_available():
    print(f"GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not detected. Please check your runtime settings.")

GPU is active: NVIDIA L4


Building the basic model.

In [6]:
# Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target'].values

# Split into training and validation sets (80/20 split)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [7]:
class FlightDataset(Dataset):
    def __init__(self, features, labels):
        # Convert to PyTorch tensors (Float32 for features, Float32 for BCEWithLogitsLoss)
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # Reshape to [N, 1]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create Datasets
train_dataset = FlightDataset(X_train_scaled, y_train)
val_dataset = FlightDataset(X_val_scaled, y_val)

# Create DataLoaders (Batch size of 1024 is good for simple models and fast GPUs)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [8]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        # Single linear layer (Input -> 1 Output)
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        # Output raw logits
        return self.linear(x)

# Initialize the model
input_dim = X_train_scaled.shape[1]
model = LogisticRegression(input_dim)

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)

LogisticRegression(
  (linear): Linear(in_features=16, out_features=1, bias=True)
)


In [9]:
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report, confusion_matrix

# Loss and Optimizer
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    train_loss = train_loss / len(train_loader.dataset)

    # Validation Loop
    model.eval()
    val_loss = 0.0

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)

            # Convert logits to probabilities via Sigmoid
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)

    # Convert lists to numpy arrays for sklearn
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # Basic AUC for the epoch
    current_auc = roc_auc_score(all_labels, all_probs)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {current_auc:.4f}")

# ==========================================
# FINAL EVALUATION & THRESHOLD OPTIMIZATION
# ==========================================
print("\n" + "="*50)
print("FINAL EVALUATION & THRESHOLD OPTIMIZATION")
print("="*50)

# AUC calculation
auc_score = roc_auc_score(all_labels, all_probs)
print(f"Validation AUC: {auc_score:.4f}")

# Find Optimal Threshold for F1-Score
print("Sweeping thresholds to maximize F1...")
best_threshold = 0.5
best_f1 = 0.0

for thresh in np.arange(0.2, 0.8, 0.02):
    preds = (all_probs >= thresh).astype(int)
    score = f1_score(all_labels, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Maximized F1:      {best_f1:.4f}")

# Final Metrics using optimized threshold
final_val_preds = (all_probs >= best_threshold).astype(int)
print(f"Final Accuracy:    {accuracy_score(all_labels, final_val_preds):.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, final_val_preds, target_names=['On-Time', 'Delayed']))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, final_val_preds))

Epoch 1/10 | Train Loss: 0.5086 | Val Loss: 0.5077 | Val AUC: 0.6456
Epoch 2/10 | Train Loss: 0.5075 | Val Loss: 0.5077 | Val AUC: 0.6456
Epoch 3/10 | Train Loss: 0.5076 | Val Loss: 0.5076 | Val AUC: 0.6457
Epoch 4/10 | Train Loss: 0.5075 | Val Loss: 0.5077 | Val AUC: 0.6449
Epoch 5/10 | Train Loss: 0.5076 | Val Loss: 0.5081 | Val AUC: 0.6440
Epoch 6/10 | Train Loss: 0.5075 | Val Loss: 0.5075 | Val AUC: 0.6451
Epoch 7/10 | Train Loss: 0.5075 | Val Loss: 0.5079 | Val AUC: 0.6445
Epoch 8/10 | Train Loss: 0.5075 | Val Loss: 0.5077 | Val AUC: 0.6448
Epoch 9/10 | Train Loss: 0.5076 | Val Loss: 0.5079 | Val AUC: 0.6443
Epoch 10/10 | Train Loss: 0.5075 | Val Loss: 0.5078 | Val AUC: 0.6459

FINAL EVALUATION & THRESHOLD OPTIMIZATION
Validation AUC: 0.6459
Sweeping thresholds to maximize F1...
Optimal Threshold: 0.20
Maximized F1:      0.4128
Final Accuracy:    0.5790

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.85      0.55      0.67   1798

In [10]:
print(df['target'].value_counts(normalize=True))

target
0    0.777745
1    0.222255
Name: proportion, dtype: float64


The model is simply learning the average. Perhaps due to noise, or lack of complexity. We will now try two approaches to solving this. A) Increasing complexity of model through additional layers. B) Reduce dimensionality and noise through PCA.

# Improving on Base Model through Complex Architecture

In [15]:
import torch.nn.functional as F

class FlightDelayDeepModel(nn.Module):
    def __init__(self, input_dim):
        super(FlightDelayDeepModel, self).__init__()

        # Layer 1: Input to Hidden (Expanding features to find interactions)
        self.fc1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64) # Normalizes layer outputs for faster training

        # Layer 2: Hidden to Hidden
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)

        # Layer 3: Hidden to Output
        self.fc3 = nn.Linear(32, 1)

        # Dropout to prevent overfitting to "on-time" noise
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Pass through Layer 1
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        # Pass through Layer 2
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        # Output Logit
        x = self.fc3(x)
        return x

# Initialize
model_complex = FlightDelayDeepModel(input_dim).to(device)

In [12]:
# Ratio of negative to positive samples
# If 78% are 0 and 22% are 1, weight = 78/22 ≈ 3.5
pos_weight_value = torch.tensor([3.5]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
optimizer = torch.optim.Adam(model_complex.parameters(), lr=0.001)

In [14]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, f1_score

num_epochs = 15

for epoch in range(num_epochs):
    model_complex.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model_complex(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    # Validation Phase
    model_complex.eval()
    val_loss = 0.0
    all_probs = [] # Store raw probabilities for AUC and Thresholding
    all_true = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            outputs = model_complex(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)

            # Get probabilities using sigmoid
            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_true.extend(batch_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader.dataset)

    # Standard 0.5 threshold for epoch-wise logging
    standard_preds = (np.array(all_probs) >= 0.5).astype(int)
    val_acc = accuracy_score(all_true, standard_preds)
    auc_score = roc_auc_score(all_true, all_probs)

    # Logging output
    print(f"\n{'='*20} Epoch {epoch+1}/{num_epochs} {'='*20}")
    print(f"Loss -> Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
    print(f"Accuracy (0.5): {val_acc:.4f} | AUC: {auc_score:.4f}")

# ==========================================
# FINAL EVALUATION & THRESHOLD OPTIMIZATION
# ==========================================
print("\n" + "="*50)
print("FINAL EVALUATION & THRESHOLD OPTIMIZATION")
print("="*50)

# Convert lists to arrays for sklearn
all_probs = np.array(all_probs)
all_true = np.array(all_true)

# Find Optimal Threshold for F1-Score
print("Sweeping thresholds to maximize F1...")
best_threshold = 0.5
best_f1 = 0.0

for thresh in np.arange(0.2, 0.8, 0.02):
    preds = (all_probs >= thresh).astype(int)
    score = f1_score(all_true, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Maximized F1:      {best_f1:.4f}")

# Final Metrics using optimized threshold
final_val_preds = (all_probs >= best_threshold).astype(int)
print(f"Final Accuracy:    {accuracy_score(all_true, final_val_preds):.4f}")

print("\nClassification Report (Optimized Threshold):")
print(classification_report(all_true, final_val_preds, target_names=['On-Time', 'Delayed'], zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_true, final_val_preds))


==================== Epoch 1/15 ====================
Loss -> Train: 1.0041 | Val: 1.0000
Accuracy (0.5): 0.6348 | AUC: 0.6776

==================== Epoch 2/15 ====================
Loss -> Train: 1.0037 | Val: 0.9998
Accuracy (0.5): 0.6361 | AUC: 0.6779

==================== Epoch 3/15 ====================
Loss -> Train: 1.0035 | Val: 0.9998
Accuracy (0.5): 0.6466 | AUC: 0.6776

==================== Epoch 4/15 ====================
Loss -> Train: 1.0031 | Val: 0.9996
Accuracy (0.5): 0.6472 | AUC: 0.6780

==================== Epoch 5/15 ====================
Loss -> Train: 1.0030 | Val: 0.9994
Accuracy (0.5): 0.6409 | AUC: 0.6782

==================== Epoch 6/15 ====================
Loss -> Train: 1.0029 | Val: 0.9994
Accuracy (0.5): 0.6480 | AUC: 0.6781

==================== Epoch 7/15 ====================
Loss -> Train: 1.0028 | Val: 0.9993
Accuracy (0.5): 0.6508 | AUC: 0.6785

==================== Epoch 8/15 ====================
Loss -> Train: 1.0027 | Val: 0.9991
Accuracy (0.5): 0.644

In [ ]:
con.close()